## Section 1). Quant_Read_Market_Expecation_Agent


In [1]:
ticker = "TSLA" 
language = "English" 

In [2]:
import json
import redis

REDIS_HOST = "redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com"
REDIS_PORT = 16376
REDIS_USERNAME = "default"
REDIS_PASSWORD = "rl8242B4UItBhFzgHW5APEqZnkYoaEZv"
COLLECTION_NAME = "Stock_Trend_INFOS"


def quant_market_expectation_read_agent(ticker: str):
    redis_key = f"{COLLECTION_NAME}:{ticker.upper()}_trends"

    client = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        username=REDIS_USERNAME,
        password=REDIS_PASSWORD,
        decode_responses=True,
    )

    data = client.get(redis_key)
    if data is None:
        print(f"No stock trend payload stored for {ticker}")
        return None

    return json.loads(data)


In [3]:
read_information = quant_market_expectation_read_agent(ticker)

### Section1 a). COT on the impaction Factors

In [4]:
# --- Section 1A • Factor Generator (keyword-only factors) ---
import json
import re
import sys
from pathlib import Path
from typing import Any, Dict, Literal, Union

from langchain.output_parsers import PydanticOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

ROOT_SENTINEL = "LLM_Call_Agent.py"


def find_repo_root(start: Path) -> Path:
    """Walk upward until the project root containing ROOT_SENTINEL is found."""
    for path in (start, *start.parents):
        if (path / ROOT_SENTINEL).exists():
            return path
    raise FileNotFoundError(f"Could not locate {ROOT_SENTINEL} upward from {start}")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from LLM_Call_Agent import LLMCallAgent  # pylint: disable=wrong-import-position

LLM_PROVIDER = Literal["deepseek", "openai"]


class FactorSet(BaseModel):
    factor_1: str = Field(description="Keyword for the top catalyst (e.g., 'Fed Rate Cut').")
    factor_2: str = Field(description="Keyword for the second catalyst.")
    factor_3: str = Field(description="Keyword for the third catalyst.")


class FactorPayload(BaseModel):
    ticker: str = Field(description="Ticker symbol in uppercase.")
    macro: FactorSet = Field(description="Macro-level catalyst keywords.")
    micro: FactorSet = Field(description="Company-level catalyst keywords.")
    sector: FactorSet = Field(description="Sector/industry catalyst keywords.")


# LangChain parser expects Pydantic v2's model_json_schema; shim it for v1.
FactorSet.model_json_schema = classmethod(lambda cls: cls.schema())
FactorPayload.model_json_schema = classmethod(lambda cls: cls.schema())

parser = PydanticOutputParser(pydantic_object=FactorPayload)

def get_system_instructions(language: str = "English") -> str:
    """Get system instructions with language support."""
    base_instructions = f"""
You are a senior equity strategist. Given the supplied stock intelligence and historical trends,
list the most impactful catalysts as short keyword-style names.

Rules:
- Output ONLY canonical event keywords (e.g., "Fed Rate Cut", "China Tariff Hike", "AI Chip Shortage").
- No directions, adjectives, or explanations—just the name of the catalyst.
- Keep each keyword under 60 characters.
- Ground every keyword in the provided context or widely known facts; never invent events.
{parser.get_format_instructions()}
""".strip()
    
    # Add language instruction if not English
    if language.lower() != "english":
        language_instruction = f"\n\nIMPORTANT: Output ALL factor names in {language} language only. Do NOT use English."
        return base_instructions + language_instruction
    else:
        return base_instructions


def _extract_json_payload(raw: str) -> str:
    """Strip markdown fences and clamp to the outermost JSON braces."""
    cleaned = raw.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start : end + 1]
    return cleaned


def build_factor_prompt(
    ticker: str, 
    read_information: Union[str, Dict[str, Any]],
    language: str = language # Add language parameter
) -> str:
    """Human prompt body sent to the LLM."""
    if isinstance(read_information, (dict, list)):
        serialized_context = json.dumps(read_information, ensure_ascii=False, indent=2)
    else:
        serialized_context = str(read_information)

    base_prompt = (
        f"Ticker: {ticker}\n\n"
        "Stock intelligence snapshot:\n"
        f"{serialized_context}\n\n"
        "Task: provide macro/micro/sector catalyst keywords that my pipeline will map directly. "
        "Return only the canonical event names."
    )
    
    # Add language instruction if not English
    if language.lower() != "english":
        language_instruction = f"\n\nIMPORTANT: Output ALL factor names in {language} language only. Do NOT use English."
        return base_prompt + language_instruction
    else:
        return base_prompt


def generate_stock_factors(
    ticker: str,
    read_information: Union[str, Dict[str, Any]],
    provider: LLM_PROVIDER = "deepseek",
    model_override: str | None = None,
    temperature: float = 0.1,
    max_tokens: int = 800,
    language: str = language  # Add language parameter
) -> FactorPayload:
    """Call the LLM (DeepSeek by default) and parse keyword factors via LangChain."""
    prompt = build_factor_prompt(ticker, read_information, language)  # Pass language to prompt
    system_instructions = get_system_instructions(language)  # Get language-specific instructions

    if provider == "deepseek":
        model = model_override or "deepseek-chat"
    else:
        provider = "openai"
        model = model_override or "gpt-4o"

    llm_agent = LLMCallAgent(default_provider=provider, default_model=model)

    if provider == "deepseek":
        raw_response = llm_agent.call_deepseek(
            prompt=prompt,
            system_message=system_instructions,  # Use language-specific instructions
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )
    else:
        raw_response = llm_agent.call_openai(
            prompt=prompt,
            system_message=system_instructions,  # Use language-specific instructions
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )

    if not raw_response:
        raise ValueError("Empty response from LLM")

    cleaned = _extract_json_payload(raw_response)
    try:
        return parser.parse(cleaned)
    except Exception as exc:
        raise ValueError(f"LLM response could not be parsed:\n{raw_response}") from exc

/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3577: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [5]:
factor_result = generate_stock_factors(ticker=ticker, read_information=read_information)
factor_result

FactorPayload(ticker='TSLA', macro=FactorSet(factor_1='Fed Rate Cut', factor_2='US-China Trade War', factor_3='Trump Auto Tariffs'), micro=FactorSet(factor_1='Robotaxi Launch', factor_2='Q3 Delivery Targets', factor_3='Musk Compensation Package'), sector=FactorSet(factor_1='EV Tax Credit Changes', factor_2='Chinese EV Competition', factor_3='Autonomous Vehicle Regulation'))

In [6]:

macro_factors = [
    factor_result.macro.factor_1,
    factor_result.macro.factor_2,
    factor_result.macro.factor_3,
]

micro_factors = [
    factor_result.micro.factor_1,
    factor_result.micro.factor_2,
    factor_result.micro.factor_3,
]

sector_factors = [
    factor_result.sector.factor_1,
    factor_result.sector.factor_2,
    factor_result.sector.factor_3,
]
print("Macro:", macro_factors)
print("Micro:", micro_factors)
print("Sector:", sector_factors)

Macro: ['Fed Rate Cut', 'US-China Trade War', 'Trump Auto Tariffs']
Micro: ['Robotaxi Launch', 'Q3 Delivery Targets', 'Musk Compensation Package']
Sector: ['EV Tax Credit Changes', 'Chinese EV Competition', 'Autonomous Vehicle Regulation']


## Section 1 b.) Retrieve the Distribution Parameter

In [7]:
# --- Section 1A • Factor Generator (keyword-only factors) ---
import json
import re
import sys
from pathlib import Path
from typing import Any, Dict, Literal, Union

from langchain.output_parsers import PydanticOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

ROOT_SENTINEL = "LLM_Call_Agent.py"


def find_repo_root(start: Path) -> Path:
    """Walk upward until the project root containing ROOT_SENTINEL is found."""
    for path in (start, *start.parents):
        if (path / ROOT_SENTINEL).exists():
            return path
    raise FileNotFoundError(f"Could not locate {ROOT_SENTINEL} upward from {start}")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from LLM_Call_Agent import LLMCallAgent  # after sys.path fix

LLM_PROVIDER = Literal["deepseek", "openai"]


class FactorSet(BaseModel):
    factor_1: str = Field(description="Keyword for the top catalyst (e.g., 'Fed Rate Cut').")
    factor_2: str = Field(description="Keyword for the second catalyst.")
    factor_3: str = Field(description="Keyword for the third catalyst.")


class FactorPayload(BaseModel):
    ticker: str = Field(description="Ticker symbol in uppercase.")
    macro: FactorSet = Field(description="Macro-level catalyst keywords.")
    micro: FactorSet = Field(description="Company-level catalyst keywords.")
    sector: FactorSet = Field(description="Sector/industry catalyst keywords.")


FactorSet.model_json_schema = classmethod(lambda cls: cls.schema())
FactorPayload.model_json_schema = classmethod(lambda cls: cls.schema())

factor_parser = PydanticOutputParser(pydantic_object=FactorPayload)

factor_system_instructions = f"""
You are a senior equity strategist. Given the supplied stock intelligence and historical trends,
list the most impactful catalysts as short keyword-style names.

Rules:
- Output ONLY canonical event keywords (e.g., "Fed Rate Cut", "China Tariff Hike", "AI Chip Shortage").
- No directions, adjectives, or explanations—just the name of the catalyst.
- Keep each keyword under 60 characters.
- Ground every keyword in the provided context or widely known facts; never invent events.
{factor_parser.get_format_instructions()}
""".strip()


def _extract_json_payload(raw: str) -> str:
    """Strip markdown fences and clamp to the outermost JSON braces."""
    cleaned = raw.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start : end + 1]
    return cleaned


def build_factor_prompt(
    ticker: str, read_information: Union[str, Dict[str, Any]]
) -> str:
    """Prompt body sent to the LLM for factor extraction."""
    if isinstance(read_information, (dict, list)):
        serialized_context = json.dumps(read_information, ensure_ascii=False, indent=2)
    else:
        serialized_context = str(read_information)

    return (
        f"Ticker: {ticker}\n\n"
        "Stock intelligence snapshot:\n"
        f"{serialized_context}\n\n"
        "Task: provide macro/micro/sector catalyst keywords that my pipeline will map directly. "
        "Return only the canonical event names."
    )


def generate_stock_factors(
    ticker: str,
    read_information: Union[str, Dict[str, Any]],
    provider: LLM_PROVIDER = "deepseek",
    model_override: str | None = None,
    temperature: float = 0.1,
    max_tokens: int = 800,
) -> FactorPayload:
    """Call the LLM (DeepSeek by default) and parse keyword factors."""
    prompt = build_factor_prompt(ticker, read_information)

    if provider == "deepseek":
        model = model_override or "deepseek-chat"
    else:
        provider = "openai"
        model = model_override or "gpt-4o"

    llm_agent = LLMCallAgent(default_provider=provider, default_model=model)

    if provider == "deepseek":
        raw_response = llm_agent.call_deepseek(
            prompt=prompt,
            system_message=factor_system_instructions,
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )
    else:
        raw_response = llm_agent.call_openai(
            prompt=prompt,
            system_message=factor_system_instructions,
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )

    if not raw_response:
        raise ValueError("Empty response from LLM")

    cleaned = _extract_json_payload(raw_response)
    try:
        return factor_parser.parse(cleaned)
    except Exception as exc:
        raise ValueError(f"LLM response could not be parsed:\n{raw_response}") from exc


# --- Section 1B • Map factors → trend keys and pull metrics ---
class FactorTrendMapping(BaseModel):
    factor_1: list[str] = Field(default_factory=list, description="Trend ids linked to factor_1")
    factor_2: list[str] = Field(default_factory=list, description="Trend ids linked to factor_2")
    factor_3: list[str] = Field(default_factory=list, description="Trend ids linked to factor_3")


class TrendMappingPayload(BaseModel):
    ticker: str = Field(description="Ticker symbol in uppercase")
    macro: FactorTrendMapping = Field(description="Macro factor mapping")
    micro: FactorTrendMapping = Field(description="Micro factor mapping")
    sector: FactorTrendMapping = Field(description="Sector factor mapping")


FactorTrendMapping.model_json_schema = classmethod(lambda cls: cls.schema())
TrendMappingPayload.model_json_schema = classmethod(lambda cls: cls.schema())

trend_mapping_parser = PydanticOutputParser(pydantic_object=TrendMappingPayload)

trend_mapping_system_message = (
    "You are a meticulous chain-of-thought analyst. Match each provided factor keyword to the trend "
    "objects in the supplied historical trend dictionary. Output only IDs that exist in the data. "
    "Avoid hallucinating keys. If nothing matches, return an empty list.\n"
    f"{trend_mapping_parser.get_format_instructions()}"
)


def build_mapping_prompt(
    ticker: str,
    factor_payload: FactorPayload,
    historical_trends: Dict[str, Any],
) -> str:
    """Prompt body to request trend-key mapping."""
    factor_summary = json.dumps(factor_payload.dict(), indent=2, ensure_ascii=False)
    trend_context = json.dumps(historical_trends, indent=2, ensure_ascii=False)

    return (
        f"Ticker: {ticker}\n\n"
        f"Factor keywords (macro/micro/sector):\n{factor_summary}\n\n"
        "Historical trend dictionary (keys, summaries, metadata):\n"
        f"{trend_context}\n\n"
        
        "Task: For each factor, list the trend ids (e.g., 'downtrend1', 'uptrend4') whose macro_reason or "
        "U need to smart map, since these factors are mix, hence dont neccesary to map all the factors to trends, some might impact by other factors rather than this factor"
        "I want need u to avoid the IPO factor"
        "micro_reason aligns with that factor. Only use keys present in the historical data. Return empty "
        "arrays when nothing matches."
    )


def map_factors_to_trends(
    llm_agent: LLMCallAgent,
    ticker: str,
    factor_payload: FactorPayload,
    historical_trends: Dict[str, Any],
    provider: LLM_PROVIDER = "deepseek",
    model_override: str | None = None,
    temperature: float = 0.0,
    max_tokens: int = 1200,
) -> TrendMappingPayload:
    """Use an LLM to map factor keywords to trend keys."""
    prompt = build_mapping_prompt(ticker, factor_payload, historical_trends)

    if provider == "deepseek":
        model = model_override or "deepseek-chat"
        raw_response = llm_agent.call_deepseek(
            prompt=prompt,
            system_message=trend_mapping_system_message,
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )
    else:
        model = model_override or "gpt-4o"
        raw_response = llm_agent.call_openai(
            prompt=prompt,
            system_message=trend_mapping_system_message,
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
        )

    if not raw_response:
        raise ValueError("Empty response from LLM during trend mapping")

    cleaned = raw_response.strip().strip("`")
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1:
        cleaned = cleaned[start : end + 1]

    try:
        return trend_mapping_parser.parse(cleaned)
    except Exception as exc:
        raise ValueError(f"Trend mapping response could not be parsed:\n{raw_response}") from exc


def _collect_metric_block(trend_entry: Dict[str, Any]) -> Dict[str, Any]:
    """Extract the metrics of interest from a single trend entry."""
    metrics = {
        "daily_average_return": trend_entry.get("day average_return"),
        "how_long_it_take": trend_entry.get("How Long it Take"),
        "return_rate_variance": trend_entry.get("return rate variance"),
    }

    if trend_entry.get("current"):
        metrics["date_range"] = trend_entry["current"]
    elif isinstance(trend_entry.get("time"), dict):
        start = trend_entry["time"].get("start")
        end = trend_entry["time"].get("end")
        metrics["date_range"] = f"{start} to {end}" if start or end else None
    else:
        metrics["date_range"] = None

    return metrics


def _locate_trend(trend_key: str, payload: Dict[str, Any]) -> tuple[Dict[str, Any] | None, str | None]:
    """Find the trend entry within current or historical collections."""
    for scope in ("current_trends", "historical_trends"):
        scope_dict = payload.get(scope) or {}
        if trend_key in scope_dict:
            return scope_dict[trend_key], scope
    return None, None


def extract_trend_metrics(
    trend_mapping: TrendMappingPayload,
    trend_payload: Dict[str, Any],
    factor_payload: FactorPayload,
) -> Dict[str, Dict[str, list[Dict[str, Any]]]]:
    """Retrieve metrics for each mapped trend key without additional LLM calls."""
    result: Dict[str, Dict[str, list[Dict[str, Any]]]] = {}
    mapping_dict = trend_mapping.dict()

    for scope in ("macro", "micro", "sector"):
        scope_map: Dict[str, list[str]] = mapping_dict.get(scope, {})  # type: ignore[arg-type]
        scope_factor_set: FactorSet = getattr(factor_payload, scope)
        scoped_metrics: Dict[str, list[Dict[str, Any]]] = {}

        for factor_name, trend_keys in scope_map.items():
            factor_label = getattr(scope_factor_set, factor_name, factor_name)
            scoped_metrics[factor_label] = []

            for trend_key in trend_keys:
                entry, collection = _locate_trend(trend_key, trend_payload)
                if not entry:
                    scoped_metrics[factor_label].append(
                        {
                            "trend_key": trend_key,
                            "section": None,
                            "error": "trend not found",
                        }
                    )
                    continue

                metric_block = _collect_metric_block(entry)
                scoped_metrics[factor_label].append(
                    {
                        "trend_key": trend_key,
                        "section": collection,
                        **metric_block,
                    }
                )

        result[scope] = scoped_metrics

    return result




In [8]:

llm_agent = LLMCallAgent(default_provider="deepseek", default_model="deepseek-chat")
trend_mapping = map_factors_to_trends(
     llm_agent=llm_agent,
     ticker=ticker,
     factor_payload=factor_result,
     historical_trends=read_information.get("historical_trends", {}),)
trend_metrics = extract_trend_metrics(trend_mapping, read_information, factor_result)
trend_metrics

{'macro': {'Fed Rate Cut': [{'trend_key': 'uptrend1',
    'section': 'historical_trends',
    'daily_average_return': 0.00921,
    'how_long_it_take': 7.0,
    'return_rate_variance': 0.00018153166584289633,
    'date_range': '2024-09-23 to 2024-09-30'},
   {'trend_key': 'uptrend3',
    'section': 'historical_trends',
    'daily_average_return': 0.07669,
    'how_long_it_take': 7.0,
    'return_rate_variance': 0.0022957069371180513,
    'date_range': '2024-11-04 to 2024-11-11'},
   {'trend_key': 'uptrend5',
    'section': 'historical_trends',
    'daily_average_return': 0.01611,
    'how_long_it_take': 13.0,
    'return_rate_variance': 0.0019366591350970003,
    'date_range': '2025-01-02 to 2025-01-15'},
   {'trend_key': 'uptrend16',
    'section': 'historical_trends',
    'daily_average_return': 0.01721,
    'how_long_it_take': 11.0,
    'return_rate_variance': 0.00019003666176140247,
    'date_range': '2025-08-01 to 2025-08-12'},
   {'trend_key': 'uptrend17',
    'section': 'historic

## Section 1 c). Caculate the weight average of events

In [9]:
from typing import Any, Dict, List
import pandas as pd

def compute_weighted_factor_metrics(
    trend_metrics: Dict[str, Dict[str, List[Dict[str, Any]]]]
) -> Dict[str, Dict[str, Dict[str, Any]]]:
    """
    Collapse trend-level metrics into factor-level aggregates using duration weights.

    Args:
        trend_metrics: Output from `extract_trend_metrics`, shaped like
            {
              "macro": {
                "<factor keyword>": [
                  {
                    "trend_key": "...",
                    "daily_average_return": float | None,
                    "how_long_it_take": float | None,
                    "return_rate_variance": float | None,
                    ...
                  },
                  ...
                ]
              },
              "micro": { ... },
              "sector": { ... }
            }

    Returns:
        {
          "macro": {
            "<factor keyword>": {
              "trend_keys": [...],
              "trend_count": int,
              "weighted_mean": float | None,
              "weighted_variance": float | None,
              "average_duration": float | None,
              "total_duration": float,
            },
            ...
          },
          "micro": { ... },
          "sector": { ... }
        }
    """
    final_metrics: Dict[str, Dict[str, Dict[str, Any]]] = {}

    for scope, factor_entries in (trend_metrics or {}).items():
        final_metrics[scope] = {}
        for factor_label, entries in (factor_entries or {}).items():
            trend_keys = []
            total_duration = 0.0
            weighted_mu_sum = 0.0
            weighted_var_sum = 0.0
            mu_weights = 0.0
            var_weights = 0.0
            duration_samples = []

            for entry in entries:
                trend_keys.append(entry.get("trend_key"))
                duration = entry.get("how_long_it_take")
                try:
                    duration = float(duration) if duration is not None else None
                except (TypeError, ValueError):
                    duration = None

                if duration is None:
                    continue

                duration_samples.append(duration)
                total_duration += duration

                mu = entry.get("daily_average_return")
                var = entry.get("return_rate_variance")

                if mu is not None:
                    weighted_mu_sum += duration * float(mu)
                    mu_weights += duration

                if var is not None:
                    weighted_var_sum += duration * float(var)
                    var_weights += duration

            weighted_mean = weighted_mu_sum / mu_weights if mu_weights else None
            weighted_variance = weighted_var_sum / var_weights if var_weights else None
            average_duration = (
                sum(duration_samples) / len(duration_samples) if duration_samples else None
            )

            final_metrics[scope][factor_label] = {
                "trend_keys": trend_keys,
                "trend_count": len(entries),
                "weighted_mean": weighted_mean,
                "weighted_variance": weighted_variance,
                "average_duration": average_duration,
                "total_duration": total_duration,
            }

    return final_metrics


def summarise_factor_metrics(
    aggregated_metrics: Dict[str, Dict[str, Dict[str, Any]]]
) -> pd.DataFrame:
    """
    Convert aggregated metrics to a summary DataFrame
    """
    rows = []
    
    for scope, factors in aggregated_metrics.items():
        for factor_name, metrics in factors.items():
            rows.append({
                'scope': scope,
                'factor': factor_name,
                'trend_keys': ', '.join(metrics.get('trend_keys', [])),
                'trend_count': metrics.get('trend_count', 0),
                'weighted_mean': metrics.get('weighted_mean'),
                'weighted_variance': metrics.get('weighted_variance'),
                'average_duration': metrics.get('average_duration'),
                'total_duration': metrics.get('total_duration', 0)
            })
    
    return pd.DataFrame(rows)

In [10]:
# --- Ultra-Simple Monte Carlo HTML Generator (Guaranteed to Work) ---
import numpy as np
import pandas as pd
import webbrowser
import os
import json
from datetime import datetime
from typing import Any, Dict, List

def create_simple_monte_carlo_html(
    df_data: pd.DataFrame,
    current_price: float,
    n_simulations: int = 1000,
    seed: int = 42,
    language: str = "English"
) -> str:
    """
    Create ultra-simple Monte Carlo HTML that definitely works
    """
    np.random.seed(seed)
    
    labels = {
        "English": {
            "title": "Monte Carlo Stress Test Analysis",
            "macro": "Macro Factors",
            "micro": "Micro Factors", 
            "sector": "Sector Factors",
            "day": "Day",
            "price": "Price ($)",
            "base_price": "Base Price"
        },
        "Chinese": {
            "title": "蒙特卡洛压力测试分析",
            "macro": "宏观因子",
            "micro": "微观因子",
            "sector": "行业因子",
            "day": "天数",
            "price": "价格 ($)",
            "base_price": "基准价格"
        }
    }
    
    current_labels = labels.get(language, labels["English"])
    
    # Generate chart data
    chart_data = {}
    scopes = df_data['scope'].unique()
    
    for scope in scopes:
        scope_data = df_data[df_data['scope'] == scope]
        chart_data[scope] = {}
        
        for _, row in scope_data.iterrows():
            factor_name = row['factor']
            mu = row['weighted_mean']
            sigma = np.sqrt(row['weighted_variance'])
            duration = int(round(row['average_duration']))
            
            # Generate price paths
            price_paths = []
            for sim in range(n_simulations):
                daily_returns = np.random.normal(mu, sigma, duration)
                price_path = current_price * np.cumprod(1 + daily_returns)
                price_paths.append(price_path)
            
            # Calculate percentiles
            price_paths_array = np.array(price_paths)
            percentiles = {
                '5th': np.percentile(price_paths_array, 5, axis=0),
                '25th': np.percentile(price_paths_array, 25, axis=0),
                '50th': np.percentile(price_paths_array, 50, axis=0),
                '75th': np.percentile(price_paths_array, 75, axis=0),
                '95th': np.percentile(price_paths_array, 95, axis=0)
            }
            
            chart_data[scope][factor_name] = {
                'percentiles': percentiles,
                'days': list(range(duration)),
                'mu': mu,
                'sigma': sigma,
                'duration': duration
            }
    
    # Start building HTML
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>{current_labels['title']}</title>
        <meta charset="UTF-8">
        <script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
        <style>
            body {{ 
                font-family: Arial, sans-serif; 
                margin: 20px; 
                background: #f5f5f5;
            }}
            .container {{ 
                max-width: 1200px; 
                margin: auto; 
                background: white; 
                padding: 20px; 
                border-radius: 10px; 
                box-shadow: 0 2px 4px rgba(0,0,0,0.1); 
            }}
            h1, h2 {{ 
                color: #2c3e50; 
                text-align: center; 
            }}
            .chart-section {{ 
                margin: 30px 0; 
                padding: 20px; 
                border: 1px solid #ddd; 
                border-radius: 8px; 
                background: #fafafa;
            }}
            .chart-container {{ 
                width: 100%; 
                height: 400px; 
                margin: 20px 0;
                position: relative;
            }}
            .stats {{ 
                display: grid; 
                grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); 
                gap: 15px; 
                margin-top: 20px;
            }}
            .stat-card {{ 
                background: white; 
                padding: 15px; 
                border-radius: 5px; 
                border-left: 4px solid #3498db;
                box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>{current_labels['title']}</h1>
            <h2>{current_labels['base_price']}: ${current_price:,.2f}</h2>
    """
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
    
    for scope_idx, scope in enumerate(scopes):
        scope_factors = chart_data[scope]
        
        html_content += f"""
            <div class="chart-section">
                <h2>{current_labels[scope]} ({len(scope_factors)} factors)</h2>
        """
        
        for factor_idx, (factor_name, factor_data) in enumerate(scope_factors.items()):
            color = colors[factor_idx % len(colors)]
            percentiles = factor_data['percentiles']
            days = factor_data['days']
            
            # Calculate statistics
            final_price = percentiles['50th'][-1]
            price_change = ((final_price - current_price) / current_price * 100)
            max_price = np.max(percentiles['95th'])
            min_price = np.min(percentiles['5th'])
            
            # Prepare data arrays
            days_array = json.dumps(days)
            p5_data = json.dumps(percentiles['5th'].tolist())
            p25_data = json.dumps(percentiles['25th'].tolist())
            p50_data = json.dumps(percentiles['50th'].tolist())
            p75_data = json.dumps(percentiles['75th'].tolist())
            p95_data = json.dumps(percentiles['95th'].tolist())
            
            html_content += f"""
                <div>
                    <h3>{factor_name}</h3>
                    <div class="chart-container">
                        <canvas id="chart_{scope_idx}_{factor_idx}"></canvas>
                    </div>
                    <div class="stats">
                        <div class="stat-card">
                            <strong>Final Price</strong><br>
                            ${final_price:.2f}
                        </div>
                        <div class="stat-card">
                            <strong>Price Change</strong><br>
                            {price_change:+.2f}%
                        </div>
                        <div class="stat-card">
                            <strong>Max Price</strong><br>
                            ${max_price:.2f}
                        </div>
                        <div class="stat-card">
                            <strong>Min Price</strong><br>
                            ${min_price:.2f}
                        </div>
                        <div class="stat-card">
                            <strong>Duration</strong><br>
                            {factor_data['duration']} days
                        </div>
                        <div class="stat-card">
                            <strong>Parameters</strong><br>
                            μ={factor_data['mu']:.4f}, σ={factor_data['sigma']:.4f}
                        </div>
                    </div>
                </div>
                
                <script>
                    document.addEventListener('DOMContentLoaded', function() {{
                        const ctx = document.getElementById('chart_{scope_idx}_{factor_idx}').getContext('2d');
                        new Chart(ctx, {{
                            type: 'line',
                            data: {{
                                labels: {days_array},
                                datasets: [
                                    {{
                                        label: '5th Percentile',
                                        data: {p5_data},
                                        borderColor: '{color}',
                                        backgroundColor: '{color}20',
                                        fill: '+1',
                                        tension: 0.4,
                                        pointRadius: 0
                                    }},
                                    {{
                                        label: '25th Percentile',
                                        data: {p25_data},
                                        borderColor: '{color}',
                                        backgroundColor: '{color}40',
                                        fill: '+1',
                                        tension: 0.4,
                                        pointRadius: 0
                                    }},
                                    {{
                                        label: '50th Percentile (Median)',
                                        data: {p50_data},
                                        borderColor: '{color}',
                                        backgroundColor: '{color}',
                                        borderWidth: 3,
                                        fill: false,
                                        tension: 0.4,
                                        pointRadius: 0
                                    }},
                                    {{
                                        label: '75th Percentile',
                                        data: {p75_data},
                                        borderColor: '{color}',
                                        backgroundColor: '{color}40',
                                        fill: '-1',
                                        tension: 0.4,
                                        pointRadius: 0
                                    }},
                                    {{
                                        label: '95th Percentile',
                                        data: {p95_data},
                                        borderColor: '{color}',
                                        backgroundColor: '{color}20',
                                        fill: '-1',
                                        tension: 0.4,
                                        pointRadius: 0
                                    }}
                                ]
                            }},
                            options: {{
                                responsive: true,
                                maintainAspectRatio: false,
                                plugins: {{
                                    title: {{
                                        display: true,
                                        text: '{factor_name} - Monte Carlo Simulation'
                                    }},
                                    legend: {{
                                        display: true,
                                        position: 'top'
                                    }}
                                }},
                                scales: {{
                                    x: {{
                                        title: {{
                                            display: true,
                                            text: '{current_labels["day"]}'
                                        }}
                                    }},
                                    y: {{
                                        title: {{
                                            display: true,
                                            text: '{current_labels["price"]}'
                                        }}
                                    }}
                                }}
                            }}
                        }});
                    }});
                </script>
            """
        
        html_content += "</div>"
    
    html_content += """
        </div>
    </body>
    </html>
    """
    
    return html_content


def display_simple_monte_carlo_html(
    df_data: pd.DataFrame,
    current_price: float,
    n_simulations: int = 1000,
    seed: int = 42,
    language: str = "English"
) -> None:
    """
    Create and display simple Monte Carlo HTML visualization
    """
    html_content = create_simple_monte_carlo_html(df_data, current_price, n_simulations, seed, language)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"monte_carlo_simple_{current_price:.0f}_{timestamp}.html"
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    webbrowser.open(f"file://{os.path.abspath(filename)}")
    
    print(f"✅ Simple Monte Carlo HTML saved as: {filename}")
    print(f"🌐 Opening in browser...")
    print(f"�� Generated {n_simulations:,} simulations per factor")


# --- Usage ---
# display_simple_monte_carlo_html(
#     df_data=summary_df,
#     current_price=238.50,
#     language="Chinese"
# )

In [11]:
aggregated_metrics = compute_weighted_factor_metrics(trend_metrics)
summary_df = summarise_factor_metrics(aggregated_metrics)
# summary_df.to_csv("TSLA_Factor_Metrics.csv", index=False)


summary_df


,scope,factor,trend_keys,trend_count,weighted_mean,weighted_variance,average_duration,total_duration
0,macro,Fed Rate Cut,"uptrend1, uptrend3, uptrend5, uptrend16, uptre...",5,0.026985,0.001117,8.600000,43.0
1,macro,US-China Trade War,"downtrend6, downtrend7, downtrend9, downtrend1...",6,-0.019449,0.001370,14.166667,85.0
2,macro,Trump Auto Tariffs,"downtrend8, uptrend10",2,0.020250,0.001398,8.000000,16.0
3,micro,Robotaxi Launch,"uptrend1, downtrend1, downtrend2, uptrend12, u...",6,0.006162,0.000727,9.666667,58.0
4,micro,Q3 Delivery Targets,"uptrend1, downtrend2, downtrend5, downtrend6, ...",5,-0.007102,0.000744,14.600000,73.0
5,micro,Musk Compensation Package,"uptrend4, uptrend16",2,0.024700,0.000487,15.500000,31.0
6,sector,EV Tax Credit Changes,"uptrend4, downtrend6, uptrend17",3,0.006748,0.000675,17.333333,52.0
7,sector,Chinese EV Competition,"downtrend2, downtrend5, downtrend6, downtrend1...",5,-0.014585,0.000931,15.200000,76.0
8,sector,Autonomous Vehicle Regulation,"uptrend7, uptrend14, uptrend15, downtrend15",4,0.010236,0.001233,9.500000,38.0


## Visulization

In [96]:
# --- Three Graphs Side by Side Monte Carlo HTML Generator ---
import numpy as np
import pandas as pd
import webbrowser
import os
import json
from datetime import datetime
from typing import Any, Dict, List

def create_three_graphs_html(
    df_data: pd.DataFrame,
    current_price: float,
    n_simulations: int = 1000,
    seed: int = 42,
    language: str = "English"
) -> str:
    """
    Create HTML with 3 graphs side by side for comparison, no event cards
    """
    np.random.seed(seed)
    
    labels = {
        "English": {
            "title": "Monte Carlo Stress Test Analysis",
            "macro": "Macro Factors",
            "micro": "Micro Factors", 
            "sector": "Sector Factors",
            "day": "Day",
            "price": "Price ($)",
            "base_price": "Base Price"
        },
        "Chinese": {
            "title": "蒙特卡洛压力测试分析",
            "macro": "宏观因子",
            "micro": "微观因子",
            "sector": "行业因子",
            "day": "天数",
            "price": "价格 ($)",
            "base_price": "基准价格"
        }
    }
    
    current_labels = labels.get(language, labels["English"])
    
    # Generate chart data
    chart_data = {}
    scopes = df_data['scope'].unique()
    
    for scope in scopes:
        scope_data = df_data[df_data['scope'] == scope]
        chart_data[scope] = {}
        
        for _, row in scope_data.iterrows():
            factor_name = row['factor']
            mu = row['weighted_mean']
            sigma = np.sqrt(row['weighted_variance'])
            duration = int(round(row['average_duration']))
            
            # Generate price paths
            price_paths = []
            for sim in range(n_simulations):
                daily_returns = np.random.normal(mu, sigma, duration)
                price_path = current_price * np.cumprod(1 + daily_returns)
                price_paths.append(price_path)
            
            # Calculate median (50th percentile) for single line
            price_paths_array = np.array(price_paths)
            median_path = np.percentile(price_paths_array, 50, axis=0)
            
            chart_data[scope][factor_name] = {
                'median_path': median_path,
                'days': list(range(duration)),
                'mu': mu,
                'sigma': sigma,
                'duration': duration
            }
    
    # Start building HTML
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>{current_labels['title']}</title>
        <meta charset="UTF-8">
        <script src="https://cdn.jsdelivr.net/npm/chart.js@2.9.4/dist/Chart.min.js"></script>
        <style>
            body {{ 
                font-family: Arial, sans-serif; 
                margin: 20px; 
                background: #f5f5f5;
            }}
            .container {{ 
                max-width: 1600px; 
                margin: auto; 
                background: white; 
                padding: 20px; 
                border-radius: 10px; 
                box-shadow: 0 2px 4px rgba(0,0,0,0.1); 
            }}
            h1 {{ 
                color: #2c3e50; 
                text-align: center; 
            }}
            .graphs-container {{
                display: grid;
                grid-template-columns: 1fr 1fr 1fr;
                gap: 20px;
                margin-top: 20px;
            }}
            .chart-section {{ 
                padding: 15px; 
                border: 1px solid #ddd; 
                border-radius: 8px; 
                background: #fafafa;
            }}
            .chart-container {{ 
                width: 100%; 
                height: 400px; 
                position: relative;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>{current_labels['title']}</h1>
            <h2 style="text-align: center; color: #666;">{current_labels['base_price']}: ${current_price:,.2f}</h2>
            
            <div class="graphs-container">
    """
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98FB98']
    
    for scope in scopes:
        scope_factors = chart_data[scope]
        
        html_content += f"""
                <div class="chart-section">
                    <div class="chart-container">
                        <canvas id="chart_{scope}" width="500" height="400"></canvas>
                    </div>
                </div>
        """
    
    html_content += """
            </div>
        </div>
    </body>
    <script>
    """
    
    # Add all JavaScript at the end
    for scope in scopes:
        scope_factors = chart_data[scope]
        
        # Prepare datasets for this scope
        datasets = []
        for factor_idx, (factor_name, factor_data) in enumerate(scope_factors.items()):
            color = colors[factor_idx % len(colors)]
            median_path = factor_data['median_path']
            
            datasets.append({
                'label': factor_name,
                'data': median_path.tolist(),
                'borderColor': color,
                'backgroundColor': color + '20',
                'fill': False,
                'tension': 0.4,
                'pointRadius': 2,
                'borderWidth': 2
            })
        
        # Get max duration for this scope
        max_duration = max([len(factor_data['days']) for factor_data in scope_factors.values()])
        days_str = json.dumps(list(range(max_duration)))
        datasets_str = json.dumps(datasets, ensure_ascii=False)
        
        html_content += f"""
        var ctx_{scope} = document.getElementById('chart_{scope}').getContext('2d');
        var chart_{scope} = new Chart(ctx_{scope}, {{
            type: 'line',
            data: {{
                labels: {days_str},
                datasets: {datasets_str}
            }},
            options: {{
                responsive: true,
                maintainAspectRatio: false,
                plugins: {{
                    title: {{
                        display: true,
                        text: '{current_labels[scope]}',
                        font: {{
                            size: 16,
                            weight: 'bold'
                        }}
                    }},
                    legend: {{
                        display: true,
                        position: 'top',
                        labels: {{
                            font: {{
                                size: 12
                            }}
                        }}
                    }}
                }},
                scales: {{
                    x: {{
                        title: {{
                            display: true,
                            text: '{current_labels["day"]}'
                        }}
                    }},
                    y: {{
                        title: {{
                            display: true,
                            text: '{current_labels["price"]}'
                        }}
                    }}
                }}
            }}
        }});
        """
    
    html_content += """
    </script>
    </html>
    """
    
    return html_content


def display_three_graphs_html(
    df_data: pd.DataFrame,
    current_price: float,
    n_simulations: int = 1000,
    seed: int = 42,
    language: str = "English"
) -> None:
    """
    Create and display HTML with 3 graphs side by side
    """
    html_content = create_three_graphs_html(df_data, current_price, n_simulations, seed, language)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"three_graphs_{current_price:.0f}_{timestamp}.html"
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    webbrowser.open(f"file://{os.path.abspath(filename)}")
    
    print(f"✅ Three Graphs HTML saved as: {filename}")
    print(f"🌐 Opening in browser...")
    print(f"📊 Generated {n_simulations:,} simulations per factor")



In [97]:

# --- Usage ---
display_three_graphs_html(
    df_data=summary_df,
    current_price=432,
    language=language
)

✅ Three Graphs HTML saved as: three_graphs_432_20250918_151614.html
🌐 Opening in browser...
📊 Generated 1,000 simulations per factor
